# ARIMA Order Selection

Grid-searches ARIMA(p,d,q) orders for each coin using walk-forward validation on the validation set.
Saves the best order per coin to `ARIMA results/arima_selected_orders.csv`, which is consumed by `03_arima_7d_rebalance_test.ipynb`.

**Run this notebook first.**

| Setting | Value |
|---------|-------|
| p, d, q ranges | p∈[0,3], d∈[0,1], q∈[0,3] → 32 combinations |
| Train / Val / Test split | 60% / 20% / 20% |
| Rebalance interval | 7 days |
| Selection metric | Validation RMSE |
| Parallelism | All 32 orders evaluated in parallel per coin (`joblib`, `n_jobs=-1`) |

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
import time
from joblib import Parallel, delayed

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.simplefilter("ignore", ConvergenceWarning)
warnings.simplefilter("ignore", FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Paths and settings

In [ ]:
data_dir   = "klines csv data/prices_cleaned"
output_dir = "ARIMA results"
os.makedirs(output_dir, exist_ok=True)

BASE_DATE      = pd.Timestamp("2022-04-01")
REBAL_INTERVAL = 7
TRAIN_RATIO    = 0.6
VAL_RATIO      = 0.2
TEST_RATIO     = 0.2
N_JOBS         = -1   # -1 = use all CPU cores

P_VALUES = range(0, 4)
D_VALUES = range(0, 2)
Q_VALUES = range(0, 4)
candidate_orders = [(p, d, q) for p in P_VALUES for d in D_VALUES for q in Q_VALUES]
print(f"Candidate orders: {len(candidate_orders)}, N_JOBS: {N_JOBS}")

## Helper functions

In [ ]:
def parse_mixed_time(series, base_date):
    """Handle mixed Binance timestamp formats (relative seconds, Unix seconds, Unix milliseconds)."""
    s = pd.to_numeric(series, errors="coerce")
    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    mask_rel = s.notna() & (s < 10**9)
    mask_sec = s.notna() & (s >= 10**9) & (s < 10**12)
    mask_ms  = s.notna() & (s >= 10**12)
    if mask_rel.any():
        parsed.loc[mask_rel] = base_date + pd.to_timedelta(s.loc[mask_rel], unit="s")
    if mask_sec.any():
        parsed.loc[mask_sec] = pd.to_datetime(s.loc[mask_sec], unit="s", errors="coerce")
    if mask_ms.any():
        parsed.loc[mask_ms] = pd.to_datetime(s.loc[mask_ms], unit="ms", errors="coerce")
    return parsed


def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    if "close" not in df.columns or "time" not in df.columns:
        raise ValueError(f"{os.path.basename(file_path)} must contain 'close' and 'time' columns")
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df["time"]  = pd.to_numeric(df["time"],  errors="coerce")
    df = df.dropna(subset=["close", "time"]).copy()
    df["parsed_time"] = parse_mixed_time(df["time"], BASE_DATE)
    df = df.dropna(subset=["parsed_time"]).copy()
    df = df.sort_values("parsed_time").reset_index(drop=True)
    df = df.drop_duplicates(subset=["parsed_time"], keep="last")
    df["return_1step"] = df["close"].pct_change()
    df = df.dropna(subset=["return_1step"]).copy()
    return df[["parsed_time", "close", "return_1step"]].reset_index(drop=True)


def split_series(df, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return (
        df.iloc[:train_end].copy().reset_index(drop=True),
        df.iloc[train_end:val_end].copy().reset_index(drop=True),
        df.iloc[val_end:].copy().reset_index(drop=True),
    )


def returns_to_price_path(start_price, forecast_returns):
    prices, prev = [], float(start_price)
    for r in forecast_returns:
        prev = prev * (1.0 + float(r))
        prices.append(prev)
    return np.array(prices)


def calc_metrics(actual, predicted):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2   = r2_score(actual, predicted)
    return mae, rmse, r2


def forecast_is_valid(arr, expected_len):
    arr = np.asarray(arr, dtype=float)
    return len(arr) == expected_len and not np.isnan(arr).any() and not np.isinf(arr).any()


def fit_arima_safe(series_values, order):
    return ARIMA(series_values, order=order,
                 enforce_stationarity=False, enforce_invertibility=False).fit()


def walk_forward_val_fast(pre_block, target_block, order, rebal_interval=7):
    """Fit once on pre_block, update with append(refit=False) for speed during grid search."""
    forecasts, actuals = [], []
    fitted = fit_arima_safe(pre_block["return_1step"].to_numpy(dtype=float), order)
    for idx in range(0, len(target_block), rebal_interval):
        if idx + rebal_interval > len(target_block):
            break
        last_price    = pre_block["close"].iloc[-1] if idx == 0 else target_block["close"].iloc[idx - 1]
        actual_return = (float(target_block["close"].iloc[idx + rebal_interval - 1]) / float(last_price)) - 1.0
        try:
            fc = fitted.forecast(steps=rebal_interval)
            if not forecast_is_valid(fc, rebal_interval):
                raise ValueError("invalid forecast")
            forecast_prices = returns_to_price_path(last_price, fc)
            forecasts.append((forecast_prices[-1] / float(last_price)) - 1.0)
            actuals.append(actual_return)
            fitted = fitted.append(
                target_block["return_1step"].iloc[idx: idx + rebal_interval].to_numpy(dtype=float),
                refit=False
            )
        except Exception:
            continue
    return forecasts, actuals


def evaluate_order(order, train, val, rebal_interval):
    """Evaluate a single ARIMA order on validation — designed to be called in parallel."""
    try:
        forecasts, actuals = walk_forward_val_fast(train, val, order, rebal_interval)
        if len(forecasts) < 5:
            return order, None
        val_mae, val_rmse, val_r2 = calc_metrics(actuals, forecasts)
        return order, {"order_used": order, "val_mae": val_mae, "val_rmse": val_rmse,
                       "val_r2": val_r2, "num_val_rebalances": len(forecasts)}
    except Exception:
        return order, None

## Run order selection (parallelised across orders)

In [ ]:
start_time = time.perf_counter()

usable_files = [
    f for f in sorted(os.listdir(data_dir))
    if os.path.isfile(os.path.join(data_dir, f))
    and not f.startswith(".")
    and not f.endswith((".py", ".npy", ".txt", ".ipynb", ".xlsx", ".keras", ".csv"))
]
print(f"Found {len(usable_files)} coins: {usable_files}")
print(f"Trying {len(candidate_orders)} orders per coin in parallel (n_jobs={N_JOBS})...\n")

summary_rows, skipped = [], []

for file_name in usable_files:
    print(f"Processing {file_name} ...")
    try:
        df = load_and_preprocess(os.path.join(data_dir, file_name))
        train, val, test = split_series(df, TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
        print(f"  Split -> train: {len(train)}, val: {len(val)}, test: {len(test)}")

        if len(train) < 30 or len(val) < REBAL_INTERVAL or len(test) < REBAL_INTERVAL:
            skipped.append((file_name, "not enough data after split"))
            print("  Skipped: not enough data")
            continue

        # Evaluate all orders in parallel
        results = Parallel(n_jobs=N_JOBS)(
            delayed(evaluate_order)(order, train, val, REBAL_INTERVAL)
            for order in candidate_orders
        )

        valid = [(order, res) for order, res in results if res is not None]
        if not valid:
            skipped.append((file_name, "no valid ARIMA order found"))
            print("  Skipped: no valid ARIMA order found")
            continue

        best_order, best = min(valid, key=lambda x: x[1]["val_rmse"])
        summary_rows.append({
            "crypto":             file_name,
            "order_used":         best["order_used"],
            "train_size":         len(train),
            "val_size":           len(val),
            "test_size":          len(test),
            "num_val_rebalances": best["num_val_rebalances"],
            "val_mae":            best["val_mae"],
            "val_rmse":           best["val_rmse"],
            "val_r2":             best["val_r2"],
        })
        print(f"  Best order: {best['order_used']}  |  Val RMSE: {best['val_rmse']:.8f}")

    except Exception as e:
        skipped.append((file_name, str(e)))
        print(f"  Failed: {e}")

summary_df = pd.DataFrame(summary_rows)
if len(summary_df) > 0:
    summary_df = summary_df.sort_values("val_rmse")
    summary_df.to_csv(os.path.join(output_dir, "arima_selected_orders.csv"), index=False)
    print(f"\nSaved selected orders to: {output_dir}/arima_selected_orders.csv")
    display(summary_df)

if skipped:
    pd.DataFrame(skipped, columns=["crypto", "reason"]).to_csv(
        os.path.join(output_dir, "arima_order_selection_skipped.csv"), index=False)
    print("Skipped:", skipped)

print(f"\nDone. Runtime: {time.perf_counter() - start_time:.2f}s")